# World model for brain MRI on OASIS-2 (Google Colab)

Runs the whole project on OASIS-2: download, preprocessing, world model, classifiers, attacks and defences.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. You need a Google account with about **3 GB free on Google Drive** (only preprocessed data, checkpoints and results go there; the 18 GB raw download stays on Colab's temporary disk).
3. You must have accepted the OASIS Data Use Terms. Do not share the data or put it in a public place.

**If Colab disconnects:** run cells 1, 2 and 6 again. Finished steps are skipped, preprocessing skips finished scans and the world model continues from its last epoch.

## 1. Connect Google Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/wm'
if not os.path.exists(REPO):
    !git clone -q https://github.com/SUDHAVALLI-17/World-model-brain-mri-.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt

## 2. Settings (edit the Drive folder name if you like)

In [ ]:
DRIVE = '/content/drive/MyDrive/world_model_oasis2'   # results live here
RAW   = '/content/oas2'                               # temporary raw data (lost on disconnect - that's fine)
os.makedirs(DRIVE, exist_ok=True)
os.environ.update({
    'DATA':   f'{DRIVE}/data',      # preprocessed volumes (~1.3 GB, encrypted)
    'RUNS':   f'{DRIVE}/runs',      # checkpoints and results
    'MODELS': f'{DRIVE}/models',    # packaged model for the web app
    'KEY':    f'{DRIVE}/data.key',  # encryption key - keep it private
    'OASIS2_DIR': RAW,
})
print('Drive folder:', DRIVE)

## 3. Download OASIS-2 (only needed until preprocessing has finished)

On the OASIS-2 page, right-click **OAS2_RAW_PART1.tar.gz** → *Copy link address*, paste it below, and do the same for PART2.
Only the first MPRAGE of each session (`mpr-1`) is extracted, and each archive is deleted afterwards, so the disk never fills up.

In [ ]:
PART1_URL = 'PASTE LINK TO OAS2_RAW_PART1.tar.gz'
PART2_URL = 'PASTE LINK TO OAS2_RAW_PART2.tar.gz'

already = os.path.exists(f"{os.environ['DATA']}/.done/preprocess")
if already:
    print('Preprocessing already finished in an earlier session - no download needed.')
else:
    os.makedirs(RAW, exist_ok=True)
    for i, url in enumerate([PART1_URL, PART2_URL], 1):
        tgz = f'/content/part{i}.tar.gz'
        !wget -q --show-progress -c "{url}" -O {tgz}
        !tar -xzf {tgz} -C {RAW} --wildcards '*mpr-1[._]*' && rm -f {tgz}
    !find {RAW} -name 'mpr-1*.hdr' | wc -l

## 4. Demographics file

Download **Demographic Data** from the OASIS-2 page (an Excel file), then upload it with the cell below (or drag it into the Drive folder once).

In [ ]:
import glob, shutil
found = glob.glob(f'{DRIVE}/*demographics*.xls*') + glob.glob(f'{DRIVE}/oasis_longitudinal*.csv')
if not found:
    from google.colab import files
    up = files.upload()
    for name in up:
        shutil.move(name, f'{DRIVE}/{name}')
    found = [f'{DRIVE}/{n}' for n in up]
os.environ['OASIS2_CSV'] = found[0]
print('demographics:', found[0])

## 5. Quick check (about 20-30 minutes) - every stage for 1-3 epochs. Its numbers mean nothing.

In [ ]:
os.environ['SMOKE'] = '1'
smoke = dict(DATA=f'{DRIVE}/smoke/data', RUNS=f'{DRIVE}/smoke/runs', MODELS=f'{DRIVE}/smoke/models')
env = ' '.join(f'{k}={v}' for k, v in smoke.items())
!{env} TRAIN_FOLDS=1 MODELSET=core bash run_all.sh 2>&1 | grep -E "^==|INFO.*(done|ep |saved|best)|ERROR|Error" | tail -60
os.environ['SMOKE'] = '0'

Look at a few preprocessing check images (brain should be upright, skull removed). If they look wrong, stop and report it.

In [ ]:
from IPython.display import Image, display
for p in sorted(glob.glob(f'{DRIVE}/smoke/data/proc/qc/*.png'))[:4]:
    print(os.path.basename(p)); display(Image(p))

## 6. Full training

Set the size to fit your GPU time. `TRAIN_FOLDS=1 MODELSET=core` is the quickest real run (a few hours on a T4);
`TRAIN_FOLDS=5 MODELSET=all` is the full paper setup (1-2 days, over several sessions - just re-run this cell after each disconnect).

In [ ]:
TRAIN_FOLDS = 1        # 1 = fold 0 only; 5 = all folds
MODELSET    = 'core'   # 'core' or 'all'
EP_PRE, EP_CLS = 100, 40

!TRAIN_FOLDS={TRAIN_FOLDS} MODELSET={MODELSET} EP_PRE={EP_PRE} EP_CLS={EP_CLS} bash run_all.sh 2>&1 | grep -E "^==|INFO.*(done|saved|best epoch|resumed)|ERROR|Error" | tail -80

## 7. Results

In [ ]:
import json, pandas as pd
R = os.environ['RUNS']
for f in ['test/test_report.json', 'robustness/robustness_report.json', 'paper1/paper1_report.json']:
    p = f'{R}/{f}'
    if os.path.exists(p):
        print('==', f); print(json.dumps(json.load(open(p)), indent=1)[:3000])
for img in ['robustness/robustness_curve.png', 'paper1/paper1_figures.png']:
    if os.path.exists(f'{R}/{img}'):
        display(Image(f'{R}/{img}'))
print('Send these folders (numbers only, no scans) for the paper:', f'{R}/test', f'{R}/robustness', f'{R}/paper1')